In [0]:
%sql
select * from `public-data`.cpdoc.dhbb_bronze where content like '%Marta Suplicy%'

In [0]:
# Instalação de dependências
%pip install databricks-vectorsearch
%pip install sentence-transformers
%pip install langchain
%pip install mlflow

# Imports
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, concat_ws, lit, row_number
from pyspark.sql.window import Window
import os

In [0]:
# PARTE 3: Chunking (Dividir em Pedaços)
# Documentos grandes precisam ser divididos em chunks menores para embedding eficiente.
# Passo 3.1: Função de Chunking

from pyspark.sql.functions import udf, explode
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

def chunk_text(text, chunk_size=500, overlap=100):
    """
    Divide texto em chunks com sobreposição
    
    Args:
        text: Texto a ser dividido
        chunk_size: Tamanho de cada chunk (caracteres)
        overlap: Sobreposição entre chunks
    
    Returns:
        Lista de chunks
    """

    if not text or len(text.strip()) == 0:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if len(chunk) > 20:
            chunks.append(chunk)
        start = end - overlap
    return chunks if chunks else []

chunk_udf = udf(chunk_text, ArrayType(StringType()))

columns=[
        "id",
        "file_name",
        "content",
        "person_name",
        "natureza",
        "sexo",
        "cargos"
    ]

df_chunked = spark.table('`public-data`.cpdoc.dhbb_bronze') \
    .select('file_name', chunk_udf('content').alias('chunks')) \
    .select('file_name', explode('chunks').alias('chunk_text'))

window = Window.orderBy('file_name')
df_chunked = df_chunked.withColumn(
    'chunk_id',
    row_number().over(window)
)

df_chunked.write.format('delta').mode('overwrite').saveAsTable(
    '`public-data`.cpdoc.dhbb_prata_chunks'
)

# print(f"✓ {df_chunked.count()} chunks criados!")
display(df_chunked)

In [0]:
# PARTE 4: Gerar Embeddings
# Passo 4.1: Usando Transformers do Hugging Face

from sentence_transformers import SentenceTransformer
import numpy as np

# Baixar e carregar modelo de embedding (small para Free tier)
#databricks-gte-large-en
model = SentenceTransformer('all-MiniLM-L6-v2')  # Pequeno e rápido

print(f"Dimensão de embeddings: {model.get_sentence_embedding_dimension()}")

# Função para gerar embedding
def generate_embedding(text):
    """Gera embedding para um texto"""
    if not text:
        return None
    embedding = model.encode(text)
    return embedding.tolist()  # Converter para lista para Spark

# Registrar como UDF
from pyspark.sql.types import ArrayType, DoubleType

embedding_udf = udf(generate_embedding, ArrayType(DoubleType()))

# Aplicar embeddings aos chunks
df_embeddings = spark.table('`public-data`.cpdoc.dhbb_prata_chunks') \
    .withColumn('embedding', embedding_udf('chunk_text'))

# Salvar com embeddings
df_embeddings.write.format('delta').mode('overwrite').saveAsTable(
    '`public-data`.cpdoc.dhbb_prata_with_embeddings'
)

print("✓ Embeddings gerados e salvos!")
df_embeddings.display()